# EfficientAD-S — efficiency only (256 × 256, pose A, global calibration)

Benchmark the **existing final checkpoint** selected for the model comparison: `efficientad_s_A_overhead_crop720_global_fixed70k_balanced__seed42__20260901_120051`.

The notebook restores the complete teacher, student, autoencoder and saved calibration statistics. It performs **no training, calibration, accuracy evaluation or image-score selection**. Input is RGB in [0,1]; normalization is internal to the PDNs/autoencoder, not applied again by the dataset loader.

Attach the extracted synthetic wheel dataset and select a **Tesla T4 GPU** in a fresh Kaggle session. The default checkpoint source is Google Drive: enable Internet and `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`, as in the TinyGLASS notebook. Alternatively attach your trusted final `model.ckpt`, choose `CHECKPOINT_SOURCE = "local"` and set its path.

No `HF_TOKEN`, ImageNet penalty dataset or external teacher download is needed. Reports are saved locally under `/kaggle/working/efficientad_efficiency/`; no checkpoint or Drive file is modified.

Energy is measured separately after the latency/memory benchmark: 5-second warm-up and three 30-second predict-only windows. It reports device-wide GPU J/image, not CPU/system/rover energy, and does not subtract idle consumption.


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import threading
import time
import warnings
from collections import Counter
from collections.abc import Iterable, Mapping
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor
from tqdm.auto import tqdm


In [ ]:
from __future__ import annotations

MODEL_NAME = "efficientad_s"
CHECKPOINT_SOURCE = "google_drive"  # google_drive | local
MODEL_CHECKPOINT_PATH = None
DRIVE_RUN_FOLDER_ID_OVERRIDE = None
FORCE_CHECKPOINT_DOWNLOAD = False
DRIVE_CHECKPOINT_CACHE_DIR = Path("/kaggle/working/efficientad_efficiency_checkpoints")
SELECTED_DRIVE_RUN = {
    "model": "efficientad_s",
    "folder_id": "1LEskJG5sIdUBHIfws1keaaZxArvD-WWU",
    "checkpoint_file_id": "1LTxIQmEovzJTnosP6Plwrobvh7qBpmr5",
    "comment": "Selected EfficientAD-S 256, pose A, crop720, global calibration, fixed 70k steps.",
}
SELECTED_SOURCE_RUN_ID = "efficientad_s_A_overhead_crop720_global_fixed70k_balanced__seed42__20260901_120051"

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_ROOT_OVERRIDE = None
OUTPUT_ROOT = Path("/kaggle/working/efficientad_efficiency")
MODEL_INPUT_SIZE = (256, 256)
EXPECTED_CHANNELS = 384
EXPECTED_MAX_STEPS = 70_000
CAMERA_POSES = ("A_overhead",)
POSE_CROPS = {"A_overhead": (90, 240, 720, 720)}
BENCHMARK_SPLIT = "validation"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_NUM_WORKERS = 0
BENCHMARK_PIN_MEMORY = False
BENCHMARK_WARMUP_RUNS = 10
BENCHMARK_TIMED_RUNS = 50
# Separate energy phase; do not derive energy from latency or GPU TDP.
ENERGY_ENABLED = True
ENERGY_METHOD = "auto"  # auto: counter first, then power sampling | counter | power
ENERGY_REPEATS = 3
ENERGY_DURATION_SECONDS = 30.0
ENERGY_WARMUP_SECONDS = 5.0
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = 0.1

RANDOM_SEED = 42
REQUIRE_CUDA = True

# Explicit compatibility with the verified 2026-09-01 final checkpoint.
# input_size is fixed by its manifest/preprocessing; mixed_precision is a
# constructor compatibility default affecting training only, not this FP32 inference.
LEGACY_CHECKPOINT_DEFAULTS = {
    "input_size": MODEL_INPUT_SIZE,
    "mixed_precision": True,
}

if CHECKPOINT_SOURCE not in {"google_drive", "local"}:
    raise ValueError("CHECKPOINT_SOURCE must be google_drive or local")


In [ ]:
from __future__ import annotations

# Installation is needed only for authenticated Drive downloads.
if CHECKPOINT_SOURCE == "google_drive":
    drive_requirements = {
        "google.oauth2.credentials": "google-auth",
        "googleapiclient": "google-api-python-client",
    }
    missing_packages = []
    for module_name, package_name in drive_requirements.items():
        try:
            installed = importlib.util.find_spec(module_name) is not None
        except ModuleNotFoundError:
            installed = False
        if not installed:
            missing_packages.append(package_name)
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [ ]:
from __future__ import annotations

# NVIDIA's maintained Python NVML bindings. This is optional sensor support;
# lack of access/install never turns a latency benchmark into a fabricated energy value.
if ENERGY_ENABLED and importlib.util.find_spec("pynvml") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"])
    except Exception as error:
        print(f"WARNING: NVML bindings could not be installed; energy may be unavailable: {error}")


## Standalone definitions

The preprocessing, dataset, loader, complete EfficientAD implementation and inference benchmark below are explicitly copied from the authoritative repository modules and checked by tests. Training definitions are retained for synchronization but are **never invoked** by the execution cells.


In [ ]:
from __future__ import annotations

RGBTriplet = tuple[float, float, float]
ImageSize = tuple[int, int]
SigmaRange = tuple[float, float]
CropBox = tuple[int, int, int, int]
IMAGENET_MEAN: RGBTriplet = (0.485, 0.456, 0.406)
IMAGENET_STD: RGBTriplet = (0.229, 0.224, 0.225)


@dataclass(frozen=True)
class PreprocessingConfig:
    """Optional preprocessing and train-time image augmentation settings.

    ``resize`` follows the torchvision convention ``(height, width)``. Every
    scalar augmentation value is disabled when ``None`` and otherwise denotes
    its maximum intensity. Photometric transforms affect RGB only.
    """

    resize: ImageSize | None = None
    resize_shorter_side: int | None = None
    center_crop: ImageSize | None = None
    pose_crop_enabled: bool = False
    pose_crops: Mapping[str, CropBox | None] = field(default_factory=dict)
    normalize_mean: RGBTriplet | None = None
    normalize_std: RGBTriplet | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: SigmaRange | None = None

    def __post_init__(self) -> None:
        if self.resize is not None and self.resize_shorter_side is not None:
            raise ValueError("resize and resize_shorter_side are mutually exclusive")
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if self.resize_shorter_side is not None and self.resize_shorter_side < 1:
            raise ValueError("resize_shorter_side must be positive")
        if self.center_crop is not None:
            if len(self.center_crop) != 2 or any(value < 1 for value in self.center_crop):
                raise ValueError("center_crop must be a positive (height, width) pair")
        for pose, crop in self.pose_crops.items():
            if not pose:
                raise ValueError("pose crop names cannot be empty")
            if crop is None:
                continue
            if len(crop) != 4:
                raise ValueError("pose crops must be (top, left, height, width)")
            if crop[2] < 1 or crop[3] < 1:
                raise ValueError("pose crop height and width must be positive")
            if crop[2] != crop[3]:
                raise ValueError("pose crops must be square")

        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")

        for name in (
            "brightness",
            "contrast",
            "gamma",
            "saturation",
            "sensor_noise",
            "gaussian_noise",
        ):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")

        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")

        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")


class WheelPreprocessor:
    """Apply aligned geometric transforms and RGB-only photometric transforms."""

    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    @staticmethod
    def _odd_kernel_size(maximum_sigma: float) -> int:
        radius = max(1, round(3.0 * maximum_sigma))
        return 2 * radius + 1

    def __call__(
        self,
        image: torch.Tensor,
        target_mask: torch.Tensor,
        anomaly_mask: torch.Tensor,
        *,
        camera_pose: str | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        config = self.config

        if config.pose_crop_enabled:
            if camera_pose is None:
                raise ValueError(
                    "camera_pose is required when pose cropping is enabled"
                )
            if camera_pose not in config.pose_crops:
                raise ValueError(f"No pose crop configured for {camera_pose!r}")
            crop = config.pose_crops[camera_pose]
            if crop is not None:
                top, left, height, width = crop
                image_height, image_width = image.shape[-2:]
                if height > image_height or width > image_width:
                    raise ValueError(
                        f"Pose crop {crop!r} for {camera_pose!r} cannot fit "
                        f"inside the input image {(image_height, image_width)!r} "
                        "without padding"
                    )
                top = min(max(top, 0), image_height - height)
                left = min(max(left, 0), image_width - width)
                image = TF.crop(image, top, left, height, width)
                target_mask = TF.crop(target_mask, top, left, height, width)
                anomaly_mask = TF.crop(anomaly_mask, top, left, height, width)

        if config.resize is not None:
            image = TF.resize(
                image,
                config.resize,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
        elif config.resize_shorter_side is not None:
            image = TF.resize(
                image,
                config.resize_shorter_side,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )

        if config.center_crop is not None:
            image = TF.center_crop(image, config.center_crop)
            target_mask = TF.center_crop(target_mask, config.center_crop)
            anomaly_mask = TF.center_crop(anomaly_mask, config.center_crop)

        # Every model-facing split uses one stable numerical contract.  Keeping
        # uint8 for evaluation while train-time augmentation returned float32
        # would either fail in the backbone or change the input scale by 255x.
        image = TF.convert_image_dtype(image, torch.float32)

        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))

            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = self._odd_kernel_size(maximum)
                image = TF.gaussian_blur(image, kernel_size, sigma)

            if config.sensor_noise is not None and config.sensor_noise > 0:
                signal_scale = image.clamp(0.0, 1.0).sqrt()
                image = image + torch.randn_like(image) * signal_scale * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise

        image = image.clamp(0.0, 1.0)
        if config.normalize_mean is not None:
            image = TF.normalize(
                image,
                mean=config.normalize_mean,
                std=config.normalize_std,
            )

        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        """Return a float RGB image in [0, 1], undoing normalization if needed."""
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)

        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
from __future__ import annotations

REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
    "camera_pose",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel Kaggle dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
        camera_poses: Sequence[str] | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = set(reader.fieldnames or [])
            missing = REQUIRED_COLUMNS - columns
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        if isinstance(camera_poses, str):
            raise TypeError("camera_poses must be a sequence of pose names, not a string")
        if camera_poses is None:
            self.camera_poses = None
        else:
            self.camera_poses = tuple(dict.fromkeys(str(pose) for pose in camera_poses))
            if not self.camera_poses or any(not pose for pose in self.camera_poses):
                raise ValueError("camera_poses must contain at least one non-empty pose")
            available_poses = {row.get("camera_pose", "") for row in rows}
            unknown_poses = set(self.camera_poses) - available_poses
            if unknown_poses:
                raise ValueError(
                    f"Unknown camera poses: {sorted(unknown_poses)}; "
                    f"available poses: {sorted(available_poses)}"
                )

        self.rows = [
            row
            for row in rows
            if row["split"] == split
            and (
                self.camera_poses is None
                or row.get("camera_pose") in self.camera_poses
            )
        ]

        if not self.rows:
            raise ValueError(
                f"No samples found for split {split!r} and "
                f"camera_poses={self.camera_poses!r}"
            )

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        """Validate the complete manifest before selecting one split."""
        if not rows:
            raise ValueError("samples.csv contains no samples")

        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)

            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )

            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])

            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image,
            target_mask,
            anomaly_mask,
            camera_pose=row.get("camera_pose"),
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }


In [ ]:
from __future__ import annotations

# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]

def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
    camera_poses: tuple[str, ...] | None = None,
) -> DataLoader:
    """Build one loader; only the training split is shuffled."""
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(
        root,
        split,
        preprocessing=preprocessing,
        camera_poses=camera_poses,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


In [ ]:
from __future__ import annotations

CHECKPOINT_SCHEMA_VERSION = 2


@dataclass(frozen=True)
class AnomalyPrediction:
    """Model-independent anomaly output with normalized scores."""

    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self) -> None:
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (
            ("anomaly_score", self.anomaly_score),
            ("anomaly_map", self.anomaly_map),
        ):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    """Small common interface shared by all anomaly detectors."""

    @property
    def is_fitted(self) -> bool:
        raise NotImplementedError

    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
        validation_loader: Iterable[Mapping[str, Any]] | None = None,
        work_dir: str | Path | None = None,
        resume: bool = False,
    ) -> AnomalyDetector:
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        raise NotImplementedError

    def checkpoint_config(self) -> dict[str, Any]:
        """Return behavior-affecting constructor settings for strict reloads."""
        return {}

    def save(self, path: str | Path, *, metadata: Mapping[str, Any] | None = None) -> Path:
        """Save model state and small run metadata in one checkpoint."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "schema_version": CHECKPOINT_SCHEMA_VERSION,
                "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
                "model_config": self.checkpoint_config(),
                "model_state_dict": self.state_dict(),
                "metadata": dict(metadata or {}),
            },
            path,
        )
        return path

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        """Allow models with dynamically sized buffers to prepare before load."""

    def load(
        self,
        path: str | Path,
        *,
        map_location: torch.device | str | None = None,
    ) -> dict[str, Any]:
        """Load a trusted project checkpoint and return its metadata."""
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
            raise ValueError(
                f"Unsupported checkpoint schema {payload.get('schema_version')!r}; "
                f"expected {CHECKPOINT_SCHEMA_VERSION}"
            )
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))


In [ ]:
from __future__ import annotations

def _capture_training_rng_state(train_loader) -> dict[str, Any]:
    generator = getattr(train_loader, "generator", None)
    return {
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "train_loader_generator_state": (
            generator.get_state() if generator is not None else None
        ),
    }


def _restore_training_rng_state(checkpoint: dict[str, Any], train_loader) -> None:
    generator_state = checkpoint.get("train_loader_generator_state")
    generator = getattr(train_loader, "generator", None)
    if generator_state is not None:
        if generator is None:
            raise TypeError("The train loader cannot restore its generator state")
        generator.set_state(
            torch.as_tensor(generator_state, dtype=torch.uint8, device="cpu")
        )
    torch.set_rng_state(
        torch.as_tensor(checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu")
    )
    if checkpoint.get("python_rng_state") is not None:
        random.setstate(checkpoint["python_rng_state"])
    cuda_rng_state = checkpoint.get("cuda_rng_state")
    if torch.cuda.is_available() and cuda_rng_state is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(state, dtype=torch.uint8, device="cpu")
            for state in cuda_rng_state
        ])


def _save_checkpoint_atomic(checkpoint: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def _clean_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
    if "image" not in batch:
        raise KeyError("Every training batch must contain 'image'")
    if torch.any(torch.as_tensor(batch.get("label", 0)) != 0):
        raise ValueError("fit accepts only clean training images")
    return batch["image"].to(device, non_blocking=True)


def _image_auc(scores: list[float], labels: list[int]) -> float:
    positives = [score for score, label in zip(scores, labels) if label == 1]
    negatives = [score for score, label in zip(scores, labels) if label == 0]
    if not positives or not negatives:
        return float("nan")
    wins = sum(p > n for p in positives for n in negatives)
    ties = sum(p == n for p in positives for n in negatives)
    return (wins + 0.5 * ties) / (len(positives) * len(negatives))


@torch.no_grad()
def _validation_auc(
    model, loader, device: torch.device, max_batches: int | None,
) -> float:
    if loader is None:
        return float("nan")
    model.eval()
    scores: list[float] = []
    labels: list[int] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        scores.extend(prediction.anomaly_score.cpu().tolist())
        labels.extend(torch.as_tensor(batch["label"]).int().tolist())
    return _image_auc(scores, labels)


class PDNSmall(nn.Module):
    """EfficientAD PDN-S, including the paper's internal ImageNet normalization."""

    def __init__(self, output_channels: int = 384) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 128, kernel_size=4, padding=3)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv2 = nn.Conv2d(128, 256, kernel_size=4, padding=3)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1)
        self.conv3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, output_channels, kernel_size=4)

    @staticmethod
    def normalize(images: torch.Tensor) -> torch.Tensor:
        mean = images.new_tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
        std = images.new_tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)
        return (images - mean) / std

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.normalize(images)
        value = F.relu(self.conv1(value))
        value = self.pool1(value)
        value = F.relu(self.conv2(value))
        value = self.pool2(value)
        value = F.relu(self.conv3(value))
        return self.conv4(value)


class EfficientADAutoencoder(nn.Module):
    """EfficientAD autoencoder with decoder geometry scaled from 256 px."""

    def __init__(self, output_channels: int = 384, output_size: int = 64) -> None:
        super().__init__()
        if output_size < 1:
            raise ValueError("output_size must be positive")
        self.output_size = int(output_size)
        scale = self.output_size / 64
        self.decoder_sizes = tuple(
            max(1, round((size + 1) * scale) - 1)
            for size in (3, 8, 15, 32, 63, 127)
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 8),
        )
        self.decoder_convs = nn.ModuleList([
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
            nn.Conv2d(64, 64, 4, padding=2),
        ])
        self.final_conv = nn.Conv2d(64, 64, 3, padding=1)
        self.output = nn.Conv2d(64, output_channels, 3, padding=1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        value = self.encoder(PDNSmall.normalize(images))
        for size, convolution in zip(self.decoder_sizes, self.decoder_convs):
            value = F.interpolate(value, size=(size, size), mode="bilinear", align_corners=False)
            value = self.dropout(F.relu(convolution(value)))
        value = F.interpolate(
            value,
            size=(self.output_size, self.output_size),
            mode="bilinear",
            align_corners=False,
        )
        return self.output(F.relu(self.final_conv(value)))


class EfficientAD(AnomalyDetector):
    """EfficientAD-S; 256 is official geometry, larger inputs scale the AE."""

    _NELSON_TO_LOCAL = {
        "0.weight": "conv1.weight", "0.bias": "conv1.bias",
        "3.weight": "conv2.weight", "3.bias": "conv2.bias",
        "6.weight": "conv3.weight", "6.bias": "conv3.bias",
        "8.weight": "conv4.weight", "8.bias": "conv4.bias",
    }

    def __init__(
        self, *, input_size: tuple[int, int] = (256, 256),
        teacher_weights_path: str | None = None,
        require_teacher_weights: bool = True, channels: int = 384,
        max_steps: int = 70_000, learning_rate: float = 1e-4,
        weight_decay: float = 1e-5, hard_quantile: float = 0.995,
        checkpoint_interval: int = 5_000, validation_interval: int = 1_000,
        early_stopping_patience: int = 8, early_stopping_min_steps: int = 0,
        early_stopping_min_relative_improvement: float = 0.001,
        lr_scheduler_patience: int = 3,
        fixed_training_duration: bool = False, mixed_precision: bool = True,
        spatial_calibration_enabled: bool = False,
        spatial_calibration_q_low: float = 0.90,
        spatial_calibration_q_high: float = 0.995,
        spatial_calibration_smoothing_sigma: float = 1.0,
        spatial_scale_floor_fraction: float = 0.25,
        static_roi_min_coverage: float = 0.50,
        image_score_topk_candidates: tuple[float, ...] = (
            0.0, 0.0005, 0.001, 0.0025, 0.005,
        ),
    ) -> None:
        super().__init__()
        input_size = tuple(int(value) for value in input_size)
        if (
            len(input_size) != 2
            or input_size[0] != input_size[1]
            or input_size[0] < 256
            or input_size[0] % 128 != 0
        ):
            raise ValueError(
                "EfficientAD input_size must be square, at least 256, "
                "and divisible by 128"
            )
        if channels < 1:
            raise ValueError("channels must be at least 1")
        if max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        if early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be at least 1")
        if not 0 <= early_stopping_min_steps <= max_steps:
            raise ValueError("early_stopping_min_steps must be in [0, max_steps]")
        if not 0 <= early_stopping_min_relative_improvement < 1:
            raise ValueError(
                "early_stopping_min_relative_improvement must be in [0, 1)"
            )
        if lr_scheduler_patience < 0:
            raise ValueError("lr_scheduler_patience cannot be negative")
        if not 0 < hard_quantile <= 1:
            raise ValueError("hard_quantile must be in (0, 1]")
        if not 0 <= spatial_calibration_q_low < spatial_calibration_q_high <= 1:
            raise ValueError(
                "spatial calibration quantiles must satisfy 0 <= low < high <= 1"
            )
        if spatial_calibration_smoothing_sigma < 0:
            raise ValueError("spatial_calibration_smoothing_sigma cannot be negative")
        if spatial_scale_floor_fraction <= 0:
            raise ValueError("spatial_scale_floor_fraction must be positive")
        if not 0 < static_roi_min_coverage <= 1:
            raise ValueError("static_roi_min_coverage must be in (0, 1]")
        image_score_topk_candidates = tuple(
            float(value) for value in image_score_topk_candidates
        )
        if (
            not image_score_topk_candidates
            or len(set(image_score_topk_candidates))
            != len(image_score_topk_candidates)
            or any(value < 0 or value > 1 for value in image_score_topk_candidates)
        ):
            raise ValueError(
                "image_score_topk_candidates must be unique fractions in [0, 1]"
            )
        native_map_size = input_size[0] // 4
        self.teacher = PDNSmall(channels)
        self.student = PDNSmall(2 * channels)
        self.autoencoder = EfficientADAutoencoder(channels, native_map_size)
        self.teacher.requires_grad_(False)
        self.input_size = input_size
        self.native_map_size = native_map_size
        self.channels = channels
        self.max_steps = max_steps
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.hard_quantile = hard_quantile
        self.checkpoint_interval = checkpoint_interval
        self.validation_interval = validation_interval
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_steps = early_stopping_min_steps
        self.early_stopping_min_relative_improvement = (
            early_stopping_min_relative_improvement
        )
        self.lr_scheduler_patience = lr_scheduler_patience
        self.fixed_training_duration = bool(fixed_training_duration)
        self.mixed_precision = bool(mixed_precision)
        self.spatial_calibration_enabled = bool(spatial_calibration_enabled)
        self.spatial_calibration_q_low = float(spatial_calibration_q_low)
        self.spatial_calibration_q_high = float(spatial_calibration_q_high)
        self.spatial_calibration_smoothing_sigma = float(
            spatial_calibration_smoothing_sigma
        )
        self.spatial_scale_floor_fraction = float(spatial_scale_floor_fraction)
        self.static_roi_min_coverage = float(static_roi_min_coverage)
        self.image_score_topk_candidates = image_score_topk_candidates
        self.teacher_source = None
        if teacher_weights_path:
            self._load_teacher(Path(teacher_weights_path))
        elif require_teacher_weights:
            raise FileNotFoundError(
                "EfficientAD-S requires the nelson1425 teacher_small.pth checkpoint"
            )
        self.register_buffer("teacher_mean", torch.zeros(1, channels, 1, 1))
        self.register_buffer("teacher_std", torch.ones(1, channels, 1, 1))
        self.register_buffer("map_st_q90", torch.tensor(0.0))
        self.register_buffer("map_st_q995", torch.tensor(1.0))
        self.register_buffer("map_ae_q90", torch.tensor(0.0))
        self.register_buffer("map_ae_q995", torch.tensor(1.0))
        native_shape = (1, 1, native_map_size, native_map_size)
        self.register_buffer("map_st_spatial_q_low", torch.zeros(native_shape))
        self.register_buffer("map_st_spatial_q_high", torch.ones(native_shape))
        self.register_buffer("map_ae_spatial_q_low", torch.zeros(native_shape))
        self.register_buffer("map_ae_spatial_q_high", torch.ones(native_shape))
        self.register_buffer("static_roi", torch.ones(native_shape, dtype=torch.bool))
        self.register_buffer("selected_spatial_calibration", torch.tensor(False))
        self.register_buffer("selected_topk_fraction", torch.tensor(0.0))
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}
        # Transient post-hoc overrides are deliberately excluded from state_dict:
        # diagnostic ablations must not alter or invalidate fitted checkpoints.
        self._diagnostic_spatial_override: bool | None = None
        self._diagnostic_static_roi_override: bool | None = None
        self._diagnostic_topk_fraction_override: float | None = None
        self._diagnostic_global_topk_pixels_override: int | None = None

    def train(self, mode: bool = True):
        super().train(mode)
        self.teacher.eval()
        return self

    def _load_teacher(self, path: Path) -> None:
        if not path.is_file():
            raise FileNotFoundError(f"EfficientAD teacher weights are missing: {path}")
        try:
            checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")
        state = checkpoint.get("state_dict", checkpoint)
        if not isinstance(state, Mapping):
            raise TypeError("The nelson1425 teacher checkpoint is not a state dict")
        if set(state) == set(self._NELSON_TO_LOCAL):
            state = {self._NELSON_TO_LOCAL[key]: value for key, value in state.items()}
        expected = set(self.teacher.state_dict())
        if set(state) != expected:
            raise ValueError(
                "Teacher checkpoint is incompatible with nelson1425 PDN-S: "
                f"expected {sorted(expected)}, got {sorted(state)}"
            )
        self.teacher.load_state_dict(state, strict=True)
        self.teacher_source = str(path)

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "input_size": self.input_size,
            "channels": self.channels,
            "max_steps": self.max_steps,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "hard_quantile": self.hard_quantile,
            "checkpoint_interval": self.checkpoint_interval,
            "validation_interval": self.validation_interval,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_steps": self.early_stopping_min_steps,
            "early_stopping_min_relative_improvement": (
                self.early_stopping_min_relative_improvement
            ),
            "lr_scheduler_patience": self.lr_scheduler_patience,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "spatial_calibration_enabled": self.spatial_calibration_enabled,
            "spatial_calibration_q_low": self.spatial_calibration_q_low,
            "spatial_calibration_q_high": self.spatial_calibration_q_high,
            "spatial_calibration_smoothing_sigma": (
                self.spatial_calibration_smoothing_sigma
            ),
            "spatial_scale_floor_fraction": self.spatial_scale_floor_fraction,
            "static_roi_min_coverage": self.static_roi_min_coverage,
            "image_score_topk_candidates": self.image_score_topk_candidates,
            "teacher_source": self.teacher_source,
        }

    def _validate_input_size(self, images: torch.Tensor, source: str) -> None:
        actual = tuple(images.shape[-2:])
        if actual != self.input_size:
            raise ValueError(
                f"EfficientAD expected {self.input_size} {source} images, got {actual}"
            )

    def _teacher(self, images: torch.Tensor) -> torch.Tensor:
        self._validate_input_size(images, "in-domain")
        value = self.teacher(images)
        return (value - self.teacher_mean) / self.teacher_std.clamp_min(1e-6)

    @staticmethod
    def _augment_autoencoder_input(images: torch.Tensor) -> torch.Tensor:
        factor = float(torch.empty(()).uniform_(0.8, 1.2))
        operation = int(torch.randint(3, ()).item())
        from torchvision.transforms import functional as transform_functional
        if operation == 0:
            return transform_functional.adjust_brightness(images, factor)
        if operation == 1:
            return transform_functional.adjust_contrast(images, factor)
        return transform_functional.adjust_saturation(images, factor)

    @torch.no_grad()
    def _raw_maps(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        teacher = self._teacher(images)
        student = self.student(images)
        autoencoder = self.autoencoder(images)
        map_st = (student[:, : self.channels] - teacher).square().mean(1, keepdim=True)
        map_ae = (
            student[:, self.channels :] - autoencoder
        ).square().mean(1, keepdim=True)
        return map_st, map_ae

    @torch.no_grad()
    def _clean_validation_loss(self, loader, device: torch.device) -> tuple[float, int]:
        """Evaluate only normal validation images with the in-domain train losses."""
        self.eval()
        total_loss = 0.0
        total_images = 0
        for batch in loader:
            labels = torch.as_tensor(batch.get("label", 0))
            clean = labels == 0
            if not clean.any():
                continue
            images = batch["image"][clean].to(device, non_blocking=True)
            teacher = self._teacher(images)
            student = self.student(images)
            distance = (student[:, : self.channels] - teacher).square()
            flattened = distance.flatten(1)
            thresholds = torch.quantile(
                flattened, self.hard_quantile, dim=1, keepdim=True
            )
            hard_mask = flattened >= thresholds
            hard_loss = (
                (flattened * hard_mask).sum(1)
                / hard_mask.sum(1).clamp_min(1)
            )
            autoencoder = self.autoencoder(images)
            student_autoencoder = student[:, self.channels :]
            autoencoder_loss = (
                (autoencoder - teacher).square().flatten(1).mean(1)
            )
            student_autoencoder_loss = (
                (student_autoencoder - autoencoder).square().flatten(1).mean(1)
            )
            per_image = hard_loss + autoencoder_loss + student_autoencoder_loss
            total_loss += float(per_image.sum())
            total_images += images.shape[0]
        if total_images == 0:
            raise ValueError(
                "early stopping requires at least one clean validation image"
            )
        return total_loss / total_images, total_images

    def _training_checkpoint(
        self, *, optimizer, scheduler, scaler, step: int, penalty_loader,
        train_epoch_generator_state=None, train_batches_into_epoch: int = 0,
        early_stopping_state: dict[str, Any] | None = None,
        optimization_complete: bool = False,
    ) -> dict[str, Any]:
        penalty_state = None
        penalty_dataset = getattr(penalty_loader, "dataset", None)
        if hasattr(penalty_dataset, "state_dict"):
            penalty_state = penalty_dataset.state_dict()
        return {
            "format": "efficientad_training_v4_spatial_calibration",
            "input_size": list(self.input_size),
            "model": self.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "grad_scaler": scaler.state_dict(),
            "step": step,
            "torch_rng_state": torch.get_rng_state(),
            "python_rng_state": random.getstate(),
            "train_epoch_generator_state": train_epoch_generator_state,
            "train_batches_into_epoch": train_batches_into_epoch,
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
            ),
            "penalty_dataset_state": penalty_state,
            "early_stopping_state": early_stopping_state,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "optimization_complete": optimization_complete,
        }

    @staticmethod
    def _save_training_checkpoint(checkpoint: dict[str, Any], path: Path) -> None:
        _save_checkpoint_atomic(checkpoint, path)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        calibration_loader=None, penalty_loader=None, work_dir=None, resume=False,
    ) -> EfficientAD:
        if penalty_loader is None:
            raise ValueError("EfficientAD training requires an ImageNet penalty_loader")
        if validation_loader is None:
            raise ValueError("EfficientAD calibration requires a normal validation_loader")
        if self.spatial_calibration_enabled and calibration_loader is None:
            raise ValueError(
                "spatial EfficientAD calibration requires a disjoint "
                "clean calibration_loader"
            )
        if self.max_steps < 1:
            raise ValueError("max_steps must be at least 1")
        device = torch.device(device)
        print(
            f"[EfficientAD] Starting fit: device={device}, "
            f"max_steps={self.max_steps:,}, resume={resume}, "
            f"training_mode={'fixed_duration' if self.fixed_training_duration else 'early_stopping'}, "
            f"validation_during_optimization={not self.fixed_training_duration}.",
            flush=True,
        )
        self.to(device)
        self.train(True)
        optimizer = torch.optim.Adam(
            [*self.student.parameters(), *self.autoencoder.parameters()],
            lr=self.learning_rate, weight_decay=self.weight_decay,
        )
        scheduler = (
            torch.optim.lr_scheduler.StepLR(
                optimizer,
                step_size=max(1, int(0.95 * self.max_steps)),
                gamma=0.1,
            )
            if self.fixed_training_duration
            else torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", factor=0.1,
                patience=self.lr_scheduler_patience, min_lr=1e-6,
                threshold=self.early_stopping_min_relative_improvement,
                threshold_mode="rel",
            )
        )
        amp_enabled = self.mixed_precision and device.type == "cuda"
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
        print(
            f"[EfficientAD] Mixed precision: "
            f"{'FP16 enabled' if amp_enabled else 'disabled'}.",
            flush=True,
        )
        checkpoint_path = (
            None if work_dir is None else Path(work_dir) / "efficientad_training.ckpt"
        )
        step = 0
        resume_train_epoch_state = None
        resume_train_batches = 0
        checkpoint_rng_state = None
        checkpoint_python_rng_state = None
        checkpoint_cuda_rng_state = None
        best_state = None
        best_validation_loss = float("inf")
        best_step = 0
        checks_without_improvement = 0
        validation_history: list[dict[str, float | int]] = []
        optimization_complete = False
        stopped_early = False
        print(
            f"[EfficientAD] Training checkpoint: "
            f"{checkpoint_path if checkpoint_path is not None else 'disabled'}",
            flush=True,
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/efficientad_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != "efficientad_training_v4_spatial_calibration":
                raise RuntimeError(
                    "This run uses disjoint spatial calibration and pose crops; "
                    "start it with RESUME=False instead of loading a legacy checkpoint."
                )
            if bool(checkpoint.get("fixed_training_duration", False)) != (
                self.fixed_training_duration
            ):
                raise RuntimeError(
                    "The EfficientAD checkpoint training-duration mode differs "
                    "from the current configuration; start a new run with resume=false."
                )
            if bool(checkpoint.get("mixed_precision", False)) != self.mixed_precision:
                raise RuntimeError(
                    "The EfficientAD checkpoint mixed-precision mode differs "
                    "from the current configuration; start a new run with resume=false."
                )
            saved_input_size = tuple(checkpoint.get("input_size", (256, 256)))
            if saved_input_size != self.input_size:
                raise RuntimeError(
                    "The EfficientAD checkpoint input size differs from the "
                    "current configuration; start a new run with resume=false."
                )
            self.load_state_dict(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            grad_scaler_state = checkpoint.get("grad_scaler")
            if grad_scaler_state:
                scaler.load_state_dict(grad_scaler_state)
            step = int(checkpoint["step"])
            early_state = checkpoint.get("early_stopping_state") or {}
            best_state = early_state.get("best_state")
            best_validation_loss = float(
                early_state.get("best_validation_loss", float("inf"))
            )
            best_step = int(early_state.get("best_step", 0))
            checks_without_improvement = int(
                early_state.get("checks_without_improvement", 0)
            )
            validation_history = list(early_state.get("validation_history", []))
            optimization_complete = bool(checkpoint.get("optimization_complete", False))
            stopped_early = bool(early_state.get("stopped_early", False))
            resume_train_epoch_state = checkpoint.get("train_epoch_generator_state")
            if resume_train_epoch_state is not None:
                resume_train_epoch_state = torch.as_tensor(
                    resume_train_epoch_state, dtype=torch.uint8, device="cpu"
                )
            resume_train_batches = int(checkpoint.get("train_batches_into_epoch", 0))
            if checkpoint.get("penalty_dataset_state") is not None:
                penalty_dataset = getattr(penalty_loader, "dataset", None)
                if not hasattr(penalty_dataset, "load_state_dict"):
                    raise TypeError("The penalty dataset cannot restore its stream state")
                penalty_dataset.load_state_dict(checkpoint["penalty_dataset_state"])
            checkpoint_rng_state = torch.as_tensor(
                checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu"
            )
            checkpoint_python_rng_state = checkpoint.get("python_rng_state")
            checkpoint_cuda_rng_state = checkpoint.get("cuda_rng_state")
            if checkpoint_cuda_rng_state is not None:
                checkpoint_cuda_rng_state = [
                    torch.as_tensor(state, dtype=torch.uint8, device="cpu")
                    for state in checkpoint_cuda_rng_state
                ]
            print(
                f"[EfficientAD] Resume state restored at step {step:,}; "
                f"train batches into epoch={resume_train_batches:,}; "
                f"optimization_complete={optimization_complete}.",
                flush=True,
            )

        if step == 0:
            print(
                "[EfficientAD] Computing teacher feature mean/std on the clean "
                "training set...",
                flush=True,
            )
            sums = torch.zeros(self.channels, device=device)
            squares = torch.zeros_like(sums)
            count = 0
            with torch.no_grad():
                for batch in tqdm(
                    train_loader, desc="EfficientAD teacher statistics",
                    unit="batch", leave=False,
                ):
                    value = self.teacher(_clean_images(batch, device))
                    sums += value.sum((0, 2, 3))
                    squares += value.square().sum((0, 2, 3))
                    count += value.shape[0] * value.shape[2] * value.shape[3]
            if count == 0:
                raise ValueError("train_loader produced no images")
            mean = sums / count
            variance = (squares / count - mean.square()).clamp_min(1e-6)
            self.teacher_mean.copy_(mean.view(1, -1, 1, 1))
            self.teacher_std.copy_(variance.sqrt().view(1, -1, 1, 1))
            print(
                f"[EfficientAD] Teacher statistics ready from {count:,} feature "
                "vectors.",
                flush=True,
            )
        else:
            print(
                "[EfficientAD] Reusing teacher statistics from the resume checkpoint.",
                flush=True,
            )

        train_generator = getattr(train_loader, "generator", None)
        if resume_train_epoch_state is not None and train_generator is not None:
            train_generator.set_state(resume_train_epoch_state)
        train_epoch_generator_state = (
            train_generator.get_state() if train_generator is not None else None
        )
        train_iterator = iter(train_loader)
        train_batches_into_epoch = 0
        if (
            resume_train_epoch_state is not None
            and step < self.max_steps
            and not optimization_complete
        ):
            for _ in range(resume_train_batches):
                try:
                    next(train_iterator)
                except StopIteration as error:
                    raise RuntimeError("Saved train-loader position is invalid") from error
                train_batches_into_epoch += 1
        elif resume_train_epoch_state is not None:
            train_batches_into_epoch = resume_train_batches
            print(
                "[EfficientAD] Optimization is already complete; skipping train-loader "
                "position replay and proceeding to calibration.",
                flush=True,
            )
        penalty_iterator = iter(penalty_loader)
        penalty_batch_seen = False
        print(
            "[EfficientAD] Local ImageNet penalty iterator created. The first "
            "next() call will fill its shuffle buffer from cached shards.",
            flush=True,
        )
        if checkpoint_rng_state is not None:
            torch.set_rng_state(checkpoint_rng_state)
            if checkpoint_python_rng_state is not None:
                random.setstate(checkpoint_python_rng_state)
            if torch.cuda.is_available() and checkpoint_cuda_rng_state is not None:
                torch.cuda.set_rng_state_all(checkpoint_cuda_rng_state)
        progress = tqdm(
            total=self.max_steps, initial=step, desc="EfficientAD-S training",
            unit="step", dynamic_ncols=True,
        )
        while step < self.max_steps and not optimization_complete:
            try:
                batch = next(train_iterator)
            except StopIteration:
                train_epoch_generator_state = (
                    train_generator.get_state() if train_generator is not None else None
                )
                train_batches_into_epoch = 0
                train_iterator = iter(train_loader)
                try:
                    batch = next(train_iterator)
                except StopIteration as error:
                    raise ValueError("train_loader produced no images") from error
            if not penalty_batch_seen:
                print(
                    "[EfficientAD] Waiting for the first ImageNet penalty batch...",
                    flush=True,
                )
            try:
                penalty_batch = next(penalty_iterator)
            except StopIteration:
                penalty_iterator = iter(penalty_loader)
                try:
                    penalty_batch = next(penalty_iterator)
                except StopIteration as error:
                    raise ValueError("penalty_loader produced no images") from error
            if not penalty_batch_seen:
                penalty_batch_seen = True
                print(
                    f"[EfficientAD] First ImageNet penalty batch ready with shape "
                    f"{tuple(penalty_batch['image'].shape)}. Training can proceed.",
                    flush=True,
                )

            train_batches_into_epoch += 1
            images = _clean_images(batch, device)
            penalty_images = penalty_batch["image"].to(device, non_blocking=True)
            self._validate_input_size(penalty_images, "ImageNet penalty")
            augmented = self._augment_autoencoder_input(images)
            train_batch_size = images.shape[0]
            penalty_batch_size = penalty_images.shape[0]
            with torch.no_grad(), torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=amp_enabled
            ):
                teacher_pair = self._teacher(torch.cat((images, augmented), dim=0))
                teacher, teacher_augmented = teacher_pair.split(
                    (train_batch_size, train_batch_size), dim=0
                )
            with torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=amp_enabled
            ):
                student_all = self.student(
                    torch.cat((images, augmented, penalty_images), dim=0)
                )
                student, student_augmented, student_penalty = student_all.split(
                    (train_batch_size, train_batch_size, penalty_batch_size), dim=0
                )
                autoencoder = self.autoencoder(augmented)

            distance = (
                student[:, : self.channels].float() - teacher.float()
            ).square()
            threshold = torch.quantile(distance.detach(), self.hard_quantile)
            loss_hard = distance[distance >= threshold].mean()
            loss_penalty = (
                student_penalty[:, : self.channels].float().square().mean()
            )
            student_autoencoder = student_augmented[:, self.channels :]
            loss_autoencoder = F.mse_loss(
                autoencoder.float(), teacher_augmented.float()
            )
            loss_student_autoencoder = F.mse_loss(
                student_autoencoder.float(), autoencoder.float()
            )
            loss = (
                loss_hard + loss_penalty + loss_autoencoder
                + loss_student_autoencoder
            )
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            step += 1
            if self.fixed_training_duration:
                scheduler.step()
            if step % 50 == 0 or step == self.max_steps:
                progress.set_postfix(
                    loss=f"{loss.detach().item():.4f}",
                    lr=f"{optimizer.param_groups[0]['lr']:.2e}", refresh=False,
                )
                progress.update(step - progress.n)

            should_stop = False
            if (
                not self.fixed_training_duration
                and (step % self.validation_interval == 0 or step == self.max_steps)
            ):
                validation_loss, validation_images = self._clean_validation_loss(
                    validation_loader, device
                )
                scheduler.step(validation_loss)
                improved = (
                    best_state is None
                    or validation_loss
                    < best_validation_loss
                    * (1.0 - self.early_stopping_min_relative_improvement)
                )
                if improved:
                    best_validation_loss = validation_loss
                    best_step = step
                    checks_without_improvement = 0
                    best_state = {
                        key: value.detach().cpu().clone()
                        for key, value in self.state_dict().items()
                    }
                else:
                    checks_without_improvement += 1
                validation_history.append({
                    "step": step,
                    "clean_validation_loss": validation_loss,
                    "clean_validation_images": validation_images,
                    "learning_rate": optimizer.param_groups[0]["lr"],
                    "improved": int(improved),
                })
                print(
                    f"[EfficientAD] Clean validation at step {step:,}: "
                    f"loss={validation_loss:.6g}, best={best_validation_loss:.6g} "
                    f"at step {best_step:,}, bad_checks="
                    f"{checks_without_improvement}/{self.early_stopping_patience}, "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}.",
                    flush=True,
                )
                self.train(True)
                should_stop = (
                    step >= self.early_stopping_min_steps
                    and checks_without_improvement >= self.early_stopping_patience
                )
                if should_stop:
                    stopped_early = True
                    print(
                        f"[EfficientAD] Early stopping at step {step:,}; "
                        f"restoring best step {best_step:,}.",
                        flush=True,
                    )

            early_stopping_state = {
                "best_state": best_state,
                "best_validation_loss": best_validation_loss,
                "best_step": best_step,
                "checks_without_improvement": checks_without_improvement,
                "validation_history": validation_history,
                "stopped_early": stopped_early,
            }
            if (
                checkpoint_path is not None
                and (
                    step % self.checkpoint_interval == 0
                    or step == self.max_steps
                    or should_stop
                )
            ):
                self._save_training_checkpoint(
                    self._training_checkpoint(
                        optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                        step=step,
                        penalty_loader=penalty_loader,
                        train_epoch_generator_state=train_epoch_generator_state,
                        train_batches_into_epoch=train_batches_into_epoch,
                        early_stopping_state=early_stopping_state,
                        optimization_complete=should_stop,
                    ),
                    checkpoint_path,
                )
                print(
                    f"[EfficientAD] Checkpoint saved at step {step:,}: "
                    f"{checkpoint_path}",
                    flush=True,
                )
            if should_stop:
                break

        if step > progress.n:
            progress.update(step - progress.n)
        progress.close()
        optimization_step = step
        if self.fixed_training_duration:
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in self.state_dict().items()
            }
            best_step = optimization_step
            print(
                f"[EfficientAD] Fixed-duration optimization complete at "
                f"step {best_step:,}; validation was not used for model selection.",
                flush=True,
            )
        else:
            if best_state is None:
                raise RuntimeError(
                    "EfficientAD completed without a validation checkpoint"
                )
            self.load_state_dict(best_state)
            print(
                f"[EfficientAD] Selected clean-validation checkpoint from step "
                f"{best_step:,} (loss={best_validation_loss:.6g}).",
                flush=True,
            )
        print(
            "[EfficientAD] Optimization complete; starting clean calibration "
            f"from {'held-out training data' if self.spatial_calibration_enabled else 'validation'}.",
            flush=True,
        )
        self.eval()
        calibration_source = (
            calibration_loader
            if self.spatial_calibration_enabled
            else validation_loader
        )
        st_maps: list[torch.Tensor] = []
        ae_maps: list[torch.Tensor] = []
        roi_masks: list[torch.Tensor] = []
        normal_images = 0
        with torch.no_grad():
            for batch in tqdm(
                calibration_source, desc="EfficientAD calibration",
                unit="batch", leave=False,
            ):
                labels = torch.as_tensor(batch.get("label", 0))
                clean = labels == 0
                if clean.any():
                    images = batch["image"][clean].to(device, non_blocking=True)
                    map_st, map_ae = self._raw_maps(images)
                    st_maps.append(map_st.cpu())
                    ae_maps.append(map_ae.cpu())
                    if self.spatial_calibration_enabled:
                        if "target_mask" not in batch:
                            raise KeyError(
                                "spatial calibration requires target_mask in every batch"
                            )
                        target_mask = batch["target_mask"][clean].float()
                        target_mask = F.interpolate(
                            target_mask,
                            size=map_st.shape[-2:],
                            mode="nearest",
                        )
                        roi_masks.append(target_mask.cpu() > 0)
                    normal_images += images.shape[0]
        if not st_maps:
            raise ValueError("calibration requires at least one normal validation image")
        st_tensor = torch.cat(st_maps).float()
        ae_tensor = torch.cat(ae_maps).float()
        st_q90 = torch.quantile(st_tensor.flatten(), 0.90)
        st_q995 = torch.maximum(
            torch.quantile(st_tensor.flatten(), 0.995), st_q90 + 1e-6
        )
        ae_q90 = torch.quantile(ae_tensor.flatten(), 0.90)
        ae_q995 = torch.maximum(
            torch.quantile(ae_tensor.flatten(), 0.995), ae_q90 + 1e-6
        )
        self.map_st_q90.copy_(st_q90.to(device))
        self.map_st_q995.copy_(st_q995.to(device))
        self.map_ae_q90.copy_(ae_q90.to(device))
        self.map_ae_q995.copy_(ae_q995.to(device))
        if self.spatial_calibration_enabled:
            spatial = self._build_spatial_calibration(
                st_tensor, ae_tensor, torch.cat(roi_masks).float()
            )
            self.map_st_spatial_q_low.copy_(spatial["st_q_low"].to(device))
            self.map_st_spatial_q_high.copy_(spatial["st_q_high"].to(device))
            self.map_ae_spatial_q_low.copy_(spatial["ae_q_low"].to(device))
            self.map_ae_spatial_q_high.copy_(spatial["ae_q_high"].to(device))
            self.static_roi.copy_(spatial["roi"].to(device))
            scoring_ablation = self._select_validation_scoring(
                validation_loader, device
            )
        else:
            scoring_ablation = {
                "selection_split": "none",
                "selected": "global_max",
                "validation": {},
            }
        print(
            f"[EfficientAD] Calibration complete on {normal_images:,} normal images: "
            f"ST(q90={st_q90.item():.6g}, q99.5={st_q995.item():.6g}), "
            f"AE(q90={ae_q90.item():.6g}, q99.5={ae_q995.item():.6g}).",
            flush=True,
        )
        self.fitted.fill_(True)
        self.fit_summary = {
            "steps": optimization_step,
            "selected_step": best_step,
            "stopped_early": stopped_early,
            "fixed_training_duration": self.fixed_training_duration,
            "mixed_precision": self.mixed_precision,
            "best_clean_validation_loss": (
                None if self.fixed_training_duration else best_validation_loss
            ),
            "validation_interval": (
                None if self.fixed_training_duration else self.validation_interval
            ),
            "early_stopping_patience": (
                None if self.fixed_training_duration else self.early_stopping_patience
            ),
            "early_stopping_min_steps": (
                None if self.fixed_training_duration else self.early_stopping_min_steps
            ),
            "early_stopping_min_relative_improvement": (
                None
                if self.fixed_training_duration
                else self.early_stopping_min_relative_improvement
            ),
            "validation_history": validation_history,
            "normal_calibration_images": normal_images,
            "calibration_pixels_per_map": st_tensor.numel(),
            "spatial_calibration_enabled": self.spatial_calibration_enabled,
            "static_roi_fraction": float(self.static_roi.float().mean()),
            "image_score_ablation": scoring_ablation,
        }
        if checkpoint_path is not None:
            self._save_training_checkpoint(
                self._training_checkpoint(
                    optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                    step=optimization_step, penalty_loader=penalty_loader,
                    train_epoch_generator_state=train_epoch_generator_state,
                    train_batches_into_epoch=train_batches_into_epoch,
                    early_stopping_state={
                        "best_state": best_state,
                        "best_validation_loss": best_validation_loss,
                        "best_step": best_step,
                        "checks_without_improvement": checks_without_improvement,
                        "validation_history": validation_history,
                        "stopped_early": stopped_early,
                    },
                    optimization_complete=True,
                ),
                checkpoint_path,
            )
        return self

    def _smooth_spatial_statistic(self, value: torch.Tensor) -> torch.Tensor:
        sigma = self.spatial_calibration_smoothing_sigma
        if sigma == 0:
            return value
        kernel = min(63, 2 * math.ceil(3 * sigma) + 1)
        if kernel % 2 == 0:
            kernel -= 1
        return gaussian_blur(value, [kernel, kernel], [sigma, sigma])

    def _build_spatial_calibration(
        self,
        st_tensor: torch.Tensor,
        ae_tensor: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        def statistics(
            values: torch.Tensor,
            global_low: torch.Tensor,
            global_high: torch.Tensor,
        ) -> tuple[torch.Tensor, torch.Tensor]:
            low = torch.quantile(
                values, self.spatial_calibration_q_low, dim=0
            ).unsqueeze(0)
            high = torch.quantile(
                values, self.spatial_calibration_q_high, dim=0
            ).unsqueeze(0)
            low = self._smooth_spatial_statistic(low)
            high = self._smooth_spatial_statistic(high)
            global_scale = (global_high - global_low).clamp_min(1e-6)
            scale_floor = global_scale * self.spatial_scale_floor_fraction
            high = low + (high - low).clamp_min(scale_floor)
            return low, high

        st_low, st_high = statistics(
            st_tensor, self.map_st_q90.cpu(), self.map_st_q995.cpu()
        )
        ae_low, ae_high = statistics(
            ae_tensor, self.map_ae_q90.cpu(), self.map_ae_q995.cpu()
        )
        roi = (
            roi_masks.mean(dim=0, keepdim=True)
            >= self.static_roi_min_coverage
        )
        if not roi.any():
            raise ValueError(
                "static ROI is empty; lower static_roi_min_coverage or audit masks"
            )
        return {
            "st_q_low": st_low,
            "st_q_high": st_high,
            "ae_q_low": ae_low,
            "ae_q_high": ae_high,
            "roi": roi,
        }

    @staticmethod
    def _calibrate_map(
        value: torch.Tensor, q90: torch.Tensor, q995: torch.Tensor,
    ) -> torch.Tensor:
        return 0.1 * (value - q90) / (q995 - q90).clamp_min(1e-6)

    def _combine_calibrated_maps(
        self,
        map_st: torch.Tensor,
        map_ae: torch.Tensor,
        *,
        spatial: bool,
    ) -> torch.Tensor:
        if spatial:
            return 0.5 * (
                self._calibrate_map(
                    map_st,
                    self.map_st_spatial_q_low,
                    self.map_st_spatial_q_high,
                )
                + self._calibrate_map(
                    map_ae,
                    self.map_ae_spatial_q_low,
                    self.map_ae_spatial_q_high,
                )
            )
        return 0.5 * (
            self._calibrate_map(map_st, self.map_st_q90, self.map_st_q995)
            + self._calibrate_map(map_ae, self.map_ae_q90, self.map_ae_q995)
        )

    def _roi_at_size(self, size: tuple[int, int]) -> torch.Tensor:
        return F.interpolate(
            self.static_roi.float(), size=size, mode="nearest"
        ).bool()

    def _aggregate_image_score(
        self,
        anomaly_map: torch.Tensor,
        *,
        use_static_roi: bool,
        topk_fraction: float,
    ) -> torch.Tensor:
        if use_static_roi:
            roi = self._roi_at_size(anomaly_map.shape[-2:]).expand(
                anomaly_map.shape[0], -1, -1, -1
            )
            values = anomaly_map.masked_select(roi).view(anomaly_map.shape[0], -1)
        else:
            values = anomaly_map.flatten(1)
        if topk_fraction == 0:
            return values.amax(1)
        k = max(1, math.ceil(values.shape[1] * topk_fraction))
        return values.topk(k, dim=1).values.mean(1)

    @staticmethod
    def _topk_candidate_name(fraction: float) -> str:
        if fraction == 0:
            return "spatial_max"
        return f"spatial_topk_{fraction:g}".replace(".", "_")

    @torch.no_grad()
    def _select_validation_scoring(
        self, validation_loader, device: torch.device,
    ) -> dict[str, Any]:
        candidates = [("global_max", False, 0.0)] + [
            (self._topk_candidate_name(fraction), True, fraction)
            for fraction in self.image_score_topk_candidates
        ]
        scores = {name: [] for name, _, _ in candidates}
        labels: list[int] = []
        self.eval()
        for batch in tqdm(
            validation_loader,
            desc="EfficientAD image-score ablation",
            unit="batch",
            leave=False,
        ):
            images = batch["image"].to(device, non_blocking=True)
            map_st, map_ae = self._raw_maps(images)
            native_maps = {
                False: self._combine_calibrated_maps(
                    map_st, map_ae, spatial=False
                ),
                True: self._combine_calibrated_maps(
                    map_st, map_ae, spatial=True
                ) * self.static_roi,
            }
            resized_maps = {
                spatial: F.interpolate(
                    value,
                    images.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )
                for spatial, value in native_maps.items()
            }
            for name, spatial, fraction in candidates:
                batch_scores = self._aggregate_image_score(
                    resized_maps[spatial],
                    use_static_roi=spatial,
                    topk_fraction=fraction,
                )
                scores[name].extend(batch_scores.cpu().tolist())
            labels.extend(torch.as_tensor(batch["label"]).int().tolist())

        validation = {
            name: {"image_auroc": _image_auc(values, labels)}
            for name, values in scores.items()
        }
        if not labels or any(
            not math.isfinite(result["image_auroc"])
            for result in validation.values()
        ):
            raise ValueError(
                "image-score selection requires both clean and anomalous "
                "validation images"
            )
        selected_name, selected_spatial, selected_fraction = max(
            candidates,
            key=lambda candidate: validation[candidate[0]]["image_auroc"],
        )
        self.selected_spatial_calibration.fill_(selected_spatial)
        self.selected_topk_fraction.fill_(selected_fraction)
        print(
            f"[EfficientAD] Validation selected {selected_name}: "
            f"image_AUROC={validation[selected_name]['image_auroc']:.6f}.",
            flush=True,
        )
        return {
            "selection_split": "validation",
            "selection_metric": "image_auroc",
            "selected": selected_name,
            "selected_spatial_calibration": selected_spatial,
            "selected_topk_fraction": selected_fraction,
            "validation": validation,
        }

    def configure_diagnostic_inference(self, mode: str) -> None:
        """Temporarily configure one calibration/ROI mode with max scoring."""
        modes = {
            "global": (False, False),
            "global_roi": (False, True),
            "spatial_roi": (True, True),
        }
        if mode not in modes:
            raise ValueError(
                f"Unknown EfficientAD diagnostic mode {mode!r}; "
                f"expected one of {tuple(modes)}"
            )
        spatial, use_static_roi = modes[mode]
        self._diagnostic_spatial_override = spatial
        self._diagnostic_static_roi_override = use_static_roi
        self._diagnostic_topk_fraction_override = 0.0
        self._diagnostic_global_topk_pixels_override = None

    def configure_diagnostic_global_topk_pixels(
        self, topk_pixels: int | None,
    ) -> None:
        """Use current resized max or a fixed number of native global pixels."""
        if topk_pixels is not None and topk_pixels < 1:
            raise ValueError("topk_pixels must be a positive integer or None")
        self._diagnostic_global_topk_pixels_override = topk_pixels

    def clear_diagnostic_inference(self) -> None:
        """Restore inference selected on validation during fit."""
        self._diagnostic_spatial_override = None
        self._diagnostic_static_roi_override = None
        self._diagnostic_topk_fraction_override = None
        self._diagnostic_global_topk_pixels_override = None

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("EfficientAD must be fitted before predict")
        map_st, map_ae = self._raw_maps(images)
        spatial = (
            bool(self.selected_spatial_calibration)
            if self._diagnostic_spatial_override is None
            else self._diagnostic_spatial_override
        )
        use_static_roi = (
            spatial
            if self._diagnostic_static_roi_override is None
            else self._diagnostic_static_roi_override
        )
        topk_fraction = (
            float(self.selected_topk_fraction)
            if self._diagnostic_topk_fraction_override is None
            else self._diagnostic_topk_fraction_override
        )
        anomaly_map = self._combine_calibrated_maps(
            map_st, map_ae, spatial=spatial
        )
        if use_static_roi:
            anomaly_map = anomaly_map * self.static_roi
        raw_map = F.interpolate(
            anomaly_map,
            images.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        normalized_map = 0.5 + torch.atan(10.0 * raw_map) / math.pi
        if use_static_roi:
            normalized_map = normalized_map * self._roi_at_size(
                images.shape[-2:]
            )
        native_topk_pixels = self._diagnostic_global_topk_pixels_override
        if native_topk_pixels is None:
            raw_score = self._aggregate_image_score(
                raw_map,
                use_static_roi=use_static_roi,
                topk_fraction=topk_fraction,
            )
        else:
            if spatial or use_static_roi:
                raise RuntimeError(
                    "Native global top-k diagnostics require global calibration "
                    "without a static ROI"
                )
            native_values = anomaly_map.flatten(1)
            if native_topk_pixels > native_values.shape[1]:
                raise ValueError(
                    "topk_pixels exceeds the number of native anomaly-map pixels"
                )
            raw_score = native_values.topk(
                native_topk_pixels, dim=1
            ).values.mean(1)
        normalized_score = 0.5 + torch.atan(10.0 * raw_score) / math.pi
        return AnomalyPrediction(normalized_score, normalized_map), raw_score, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]


In [ ]:
from __future__ import annotations

# Authenticated Google Drive download with atomic write and size verification.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"


def get_kaggle_secret(name: str, *, required: bool = False) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Missing required Kaggle secret {name!r}. Add the same Google Drive "
            "credentials used by the roadmap notebook."
        )
    return value


def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)


def parse_drive_item_id(value: str) -> str:
    from urllib.parse import parse_qs, urlparse

    value = str(value or "").strip()
    if not value:
        raise ValueError("Google Drive ID/URL cannot be empty")
    if "/" not in value and "?" not in value:
        item_id = value
    else:
        parsed = urlparse(value)
        parts = [part for part in parsed.path.split("/") if part]
        item_id = parse_qs(parsed.query).get("id", [None])[0]
        if item_id is None:
            for marker in ("d", "folders"):
                if marker in parts and parts.index(marker) + 1 < len(parts):
                    item_id = parts[parts.index(marker) + 1]
                    break
    if not item_id or not all(
        character.isalnum() or character in "-_" for character in item_id
    ):
        raise ValueError(f"Invalid Google Drive ID/URL: {value!r}")
    return item_id


def drive_children(service, parent_id: str, name: str | None = None) -> list[dict]:
    backslash = chr(92)
    escaped_parent = parent_id.replace("'", backslash + "'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace(backslash, backslash * 2).replace(
            "'", backslash + "'"
        )
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size,parents)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])


def selected_drive_run() -> dict[str, Any]:
    if DRIVE_RUN_FOLDER_ID_OVERRIDE:
        return {
            "model": MODEL_NAME,
            "folder_id": parse_drive_item_id(DRIVE_RUN_FOLDER_ID_OVERRIDE),
            "checkpoint_file_id": None,
            "compatible_with_current_preset": None,
            "comment": "Manual override; strict checkpoint configuration still applies.",
        }
    if SELECTED_DRIVE_RUN is None:
        raise RuntimeError(
            f"No verified Drive checkpoint is registered for {MODEL_NAME}. "
            "Use CHECKPOINT_SOURCE='local' and set MODEL_CHECKPOINT_PATH."
        )
    return dict(SELECTED_DRIVE_RUN)


def resolve_drive_checkpoint_file(service, run: Mapping[str, Any]) -> dict[str, Any]:
    folder_id = parse_drive_item_id(run["folder_id"])
    matches = [
        item
        for item in drive_children(service, folder_id, "model.ckpt")
        if item["mimeType"] != DRIVE_FOLDER_MIME
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one model.ckpt in Drive folder {folder_id}, "
            f"found {len(matches)}"
        )
    item = matches[0]
    expected_file_id = run.get("checkpoint_file_id")
    if expected_file_id and item["id"] != expected_file_id:
        raise ValueError(
            "Drive folder checkpoint ID changed: "
            f"expected {expected_file_id}, found {item['id']}"
        )
    return item


def download_drive_checkpoint(
    service,
    run: Mapping[str, Any],
) -> tuple[Path, dict[str, Any]]:
    from googleapiclient.http import MediaIoBaseDownload

    item = resolve_drive_checkpoint_file(service, run)
    file_id = item["id"]
    destination = DRIVE_CHECKPOINT_CACHE_DIR / MODEL_NAME / f"{file_id}.ckpt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size") or 0)
    if (
        not FORCE_CHECKPOINT_DOWNLOAD
        and destination.is_file()
        and expected_size
        and destination.stat().st_size == expected_size
    ):
        print(f"Reusing size-verified cached checkpoint: {destination}")
        return destination, {
            **item,
            "cached": True,
            "run_folder_id": parse_drive_item_id(run["folder_id"]),
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = service.files().get_media(fileId=file_id)
    try:
        with temporary.open("wb") as handle, tqdm(
            total=expected_size or None,
            desc=f"Downloading {MODEL_NAME} model.ckpt",
            unit="B",
            unit_scale=True,
            leave=False,
            dynamic_ncols=True,
        ) as progress:
            downloader = MediaIoBaseDownload(
                handle,
                request,
                chunksize=8 * 1024 * 1024,
            )
            downloaded = 0
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=5)
                if status is not None:
                    current = int(status.resumable_progress)
                    progress.update(max(0, current - downloaded))
                    downloaded = current
        if expected_size and temporary.stat().st_size != expected_size:
            raise IOError(
                f"Incomplete Drive checkpoint: {temporary.stat().st_size} / "
                f"{expected_size} bytes"
            )
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return destination, {
        **item,
        "cached": False,
        "run_folder_id": parse_drive_item_id(run["folder_id"]),
    }


def download_selected_checkpoint_from_drive() -> tuple[Path, dict[str, Any]]:
    run = selected_drive_run()
    if run.get("model") != MODEL_NAME:
        raise ValueError(
            f"Drive run model {run.get('model')!r} does not match {MODEL_NAME!r}"
        )
    service = build_drive_service()
    checkpoint_path, metadata = download_drive_checkpoint(service, run)
    print(
        f"Drive run folder: {parse_drive_item_id(run['folder_id'])} | "
        f"checkpoint file: {metadata['id']} | bytes={metadata.get('size', 'unknown')}"
    )
    return checkpoint_path, {"run": run, "file": metadata}


In [ ]:
from __future__ import annotations

def load_frozen_efficientad(
    checkpoint_path: str | Path, *, device: torch.device | str,
) -> tuple[EfficientAD, dict[str, Any]]:
    """Restore all saved branches and global calibration without fit or recalibration."""
    checkpoint_path = Path(checkpoint_path).expanduser()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Final model checkpoint is missing: {checkpoint_path}")
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    required = {"schema_version", "model_class", "model_config", "model_state_dict"}
    if not isinstance(payload, dict) or not required <= set(payload):
        raise ValueError("Expected a final model.ckpt, not a training checkpoint")
    if payload["schema_version"] != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Unsupported final checkpoint schema")
    if str(payload["model_class"]).rsplit(".", 1)[-1] != "EfficientAD":
        raise ValueError("The supplied checkpoint is not EfficientAD")
    if not isinstance(payload["model_config"], dict):
        raise ValueError("Checkpoint model_config must be a dictionary")
    original_saved_config = dict(payload["model_config"])
    saved_config = dict(original_saved_config)
    defaulted_fields = []
    for key, value in LEGACY_CHECKPOINT_DEFAULTS.items():
        if key not in saved_config:
            saved_config[key] = value
            defaulted_fields.append(key)
    canonicalized_sequences = []
    for key in ("input_size", "image_score_topk_candidates"):
        if key in saved_config and isinstance(saved_config[key], list):
            saved_config[key] = tuple(saved_config[key])
            canonicalized_sequences.append(key)

    if tuple(saved_config.get("input_size") or ()) != tuple(MODEL_INPUT_SIZE):
        raise ValueError("Use the selected 256x256 checkpoint, not a different resolution")
    if saved_config.get("channels") != EXPECTED_CHANNELS:
        raise ValueError("The selected EfficientAD-S checkpoint must use 384 channels")
    if saved_config.get("max_steps") != EXPECTED_MAX_STEPS:
        raise ValueError("Use the selected fixed-70k checkpoint")
    if saved_config.get("fixed_training_duration") is not True:
        raise ValueError("The selected checkpoint must use fixed training duration")
    if saved_config.get("spatial_calibration_enabled") is not False:
        raise ValueError("Use the selected globally calibrated checkpoint, not spatial calibration")
    if tuple(saved_config.get("image_score_topk_candidates") or ()) != (0.0,):
        raise ValueError("The selected checkpoint must use global-max image scoring")

    metadata = dict(payload.get("metadata") or {})
    preprocessing = dict(metadata.get("preprocessing") or {})
    if preprocessing:
        expected_preprocessing = {
            "input_size": list(MODEL_INPUT_SIZE),
            "resize": list(MODEL_INPUT_SIZE),
            "resize_shorter_side": None,
            "center_crop": None,
            "pose_crop_enabled": True,
            "normalize_mean": None,
            "normalize_std": None,
        }
        for key, expected in expected_preprocessing.items():
            if key in preprocessing and preprocessing[key] != expected:
                raise ValueError(f"Checkpoint preprocessing mismatch: {key}")
        saved_pose_crop = (preprocessing.get("pose_crops") or {}).get("A_overhead")
        if saved_pose_crop is not None and tuple(saved_pose_crop) != tuple(POSE_CROPS["A_overhead"]):
            raise ValueError("Checkpoint pose-A crop does not match the fixed crop720")
    dataset_metadata = dict(metadata.get("dataset") or {})
    if dataset_metadata.get("camera_poses") is not None:
        if tuple(dataset_metadata["camera_poses"]) != tuple(CAMERA_POSES):
            raise ValueError("The selected checkpoint must have been trained on pose A only")

    constructor_fields = set(inspect.signature(EfficientAD.__init__).parameters) - {"self"}
    unknown_fields = set(saved_config) - constructor_fields - {"teacher_source"}
    if unknown_fields:
        raise ValueError(f"Unrecognized EfficientAD checkpoint fields: {sorted(unknown_fields)}")
    constructor_config = {
        key: value for key, value in saved_config.items() if key in constructor_fields
    }
    # The final model_state_dict includes the full teacher; do not download/reinitialize it.
    constructor_config["teacher_weights_path"] = None
    constructor_config["require_teacher_weights"] = False
    restored = EfficientAD(**constructor_config)
    restored.teacher_source = saved_config.get("teacher_source")
    if restored.checkpoint_config() != saved_config:
        raise ValueError("Saved EfficientAD configuration does not match the current implementation exactly")
    restored._prepare_state_dict_for_load(payload["model_state_dict"])
    restored.load_state_dict(payload["model_state_dict"], strict=True)
    if not restored.is_fitted:
        raise ValueError("The supplied final checkpoint contains an unfitted model")
    if bool(restored.selected_spatial_calibration):
        raise ValueError("The saved state selected spatial calibration, not the required global mode")
    if float(restored.selected_topk_fraction) != 0.0:
        raise ValueError("The saved state does not use global-max image scoring")
    restored.fit_summary = dict(metadata.get("fit_summary") or {})
    restored.to(device).eval()
    calibration_statistics = {
        name: float(getattr(restored, name).detach().cpu())
        for name in ("map_st_q90", "map_st_q995", "map_ae_q90", "map_ae_q995")
    }
    if not all(math.isfinite(value) for value in calibration_statistics.values()):
        raise ValueError("Saved global calibration statistics must be finite")
    restore_info = {
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_size_bytes": checkpoint_path.stat().st_size,
        "checkpoint_model_class": payload["model_class"],
        "source_run_id": metadata.get("run_id"),
        "saved_model_config": original_saved_config,
        "restored_model_config": restored.checkpoint_config(),
        "legacy_defaulted_fields": defaulted_fields,
        "canonicalized_sequence_fields": canonicalized_sequences,
        "legacy_compatibility_note": (
            "Only input_size and mixed_precision can be defaulted for the verified legacy checkpoint. "
            "mixed_precision is a training-only constructor default; inference uses FP32 without autocast."
        ),
        "source_manifest_sha256": dataset_metadata.get("manifest_sha256"),
        "calibration_mode": "global",
        "image_score_mode": "global_max",
        "selected_spatial_calibration": bool(restored.selected_spatial_calibration),
        "selected_topk_fraction": float(restored.selected_topk_fraction),
        "saved_calibration_statistics": calibration_statistics,
        "teacher_source_at_training": restored.teacher_source,
        "external_input_normalization": False,
        "inference_autocast": False,
        "training_performed": False,
        "accuracy_evaluation_performed": False,
        "calibration_performed": False,
        "image_score_selection_performed": False,
    }
    return restored, restore_info


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_json_atomic(value: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)
    return path


In [ ]:
from __future__ import annotations

def _latency_summary(milliseconds: list[float]) -> dict[str, float]:
    if not milliseconds:
        raise ValueError("Latency samples cannot be empty")
    ordered = sorted(float(value) for value in milliseconds)

    def percentile(fraction: float) -> float:
        position = fraction * (len(ordered) - 1)
        lower, upper = math.floor(position), math.ceil(position)
        weight = position - lower
        return ordered[lower] * (1 - weight) + ordered[upper] * weight

    return {
        "minimum_ms": ordered[0],
        "mean_ms": sum(ordered) / len(ordered),
        "median_ms": percentile(0.50),
        "p90_ms": percentile(0.90),
        "p95_ms": percentile(0.95),
        "maximum_ms": ordered[-1],
    }


def _cuda_memory_baseline(device: torch.device) -> tuple[int, int] | None:
    if device.type != "cuda":
        return None
    torch.cuda.synchronize(device)
    torch.cuda.empty_cache()
    baseline = (
        torch.cuda.memory_allocated(device), torch.cuda.memory_reserved(device)
    )
    torch.cuda.reset_peak_memory_stats(device)
    return baseline


def _cuda_memory_report(
    device: torch.device, baseline: tuple[int, int] | None,
) -> dict[str, float] | None:
    if baseline is None:
        return None
    allocated, reserved = baseline
    peak_allocated = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)
    return {
        "baseline_allocated_mib": allocated / 2**20,
        "baseline_reserved_mib": reserved / 2**20,
        "peak_allocated_mib": peak_allocated / 2**20,
        "peak_reserved_mib": peak_reserved / 2**20,
        "incremental_peak_allocated_mib": (peak_allocated - allocated) / 2**20,
        "incremental_peak_reserved_mib": (peak_reserved - reserved) / 2**20,
    }


@torch.inference_mode()
def benchmark_anomaly_inference(
    detector: Any,
    loader: Any,
    *,
    device: torch.device | str,
    warmup_runs: int = 10,
    timed_runs: int = 50,
    split: str = "validation",
) -> dict[str, Any]:
    """Measure predict-only and sequential batch-one pipeline latency.

    ``predict_only`` reuses one preprocessed, device-resident input. The other
    measurements include the loader's complete work, which for the synthetic
    dataset includes annotation I/O and aligned mask transforms. They are not
    a mask-free deployment benchmark. CUDA allocator peaks include all live
    tensors on the selected device, not just the model, and exclude driver
    allocations. No power sensor or energy estimate is used.
    """
    if getattr(loader, "batch_size", None) != 1:
        raise ValueError("Inference benchmark loader must use batch_size=1")
    if getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Sequential inference benchmark requires num_workers=0")
    if not isinstance(warmup_runs, int) or not isinstance(timed_runs, int):
        raise TypeError("warmup_runs and timed_runs must be integers")
    if warmup_runs < 1 or timed_runs < 1:
        raise ValueError("warmup_runs and timed_runs must be positive")
    if not detector.is_fitted:
        raise RuntimeError("Fit or restore the detector before benchmarking")

    device = torch.device(device)
    uses_cuda = device.type == "cuda"
    was_training = detector.training
    detector.to(device)
    detector.eval()

    def synchronize() -> None:
        if uses_cuda:
            torch.cuda.synchronize(device)

    def next_batch(iterator):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            try:
                return next(iterator), iterator
            except StopIteration as error:
                raise ValueError("Inference benchmark loader is empty") from error

    try:
        iterator = iter(loader)
        batch, iterator = next_batch(iterator)
        images = batch["image"].to(device, non_blocking=uses_cuda)
        if images.ndim != 4 or images.shape[0] != 1:
            raise ValueError("Inference benchmark requires one BCHW image")
        input_shape, input_dtype = list(images.shape), str(images.dtype)
        for _ in range(warmup_runs):
            prediction = detector.predict(images)
            del prediction
        synchronize()
        baseline = _cuda_memory_baseline(device)
        predict_ms = []
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            prediction = detector.predict(images)
            synchronize()
            predict_ms.append((time.perf_counter() - start) * 1_000)
            # Do not retain the previous output across the next forward.
            del prediction
        predict_memory = _cuda_memory_report(device, baseline)
        del images, batch, iterator

        iterator = iter(loader)
        for _ in range(warmup_runs):
            batch, iterator = next_batch(iterator)
            images = batch["image"].to(device, non_blocking=uses_cuda)
            prediction = detector.predict(images)
            synchronize()
            del prediction, images, batch
        baseline = _cuda_memory_baseline(device)
        preprocessing_ms, accelerator_ms, pipeline_ms = [], [], []
        iterator = iter(loader)
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            batch, iterator = next_batch(iterator)
            preprocessing_end = time.perf_counter()
            images = batch["image"].to(device, non_blocking=uses_cuda)
            if list(images.shape) != input_shape or str(images.dtype) != input_dtype:
                raise ValueError("Benchmark inputs must have one stable shape and dtype")
            prediction = detector.predict(images)
            synchronize()
            end = time.perf_counter()
            preprocessing_ms.append((preprocessing_end - start) * 1_000)
            accelerator_ms.append((end - preprocessing_end) * 1_000)
            pipeline_ms.append((end - start) * 1_000)
            del prediction, images, batch
        pipeline_memory = _cuda_memory_report(device, baseline)

        predict_summary = _latency_summary(predict_ms)
        accelerator_summary = _latency_summary(accelerator_ms)
        pipeline_summary = _latency_summary(pipeline_ms)
        parameters = list(detector.parameters())
        buffers = list(detector.buffers())
        bank = getattr(detector, "memory_bank", None)
        return {
            "schema_version": 1,
            "scope": (
                "predict_only: one device-resident preprocessed image; pipeline: "
                "sequential loader (including annotation I/O and mask transforms), "
                "host-to-device transfer and detector.predict; no worker prefetch"
            ),
            "memory_scope": (
                "CUDA allocator totals include the model, memory bank and other live "
                "tensors on this device; driver allocations are excluded"
            ),
            "split": split,
            "batch_size": 1,
            "warmup_runs": warmup_runs,
            "timed_runs": timed_runs,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device) if uses_cuda else None,
            "input_shape": input_shape,
            "input_dtype": input_dtype,
            "model": type(detector).__name__,
            "gaussian_sigma": getattr(detector, "gaussian_sigma", None),
            "total_parameters": sum(parameter.numel() for parameter in parameters),
            "trainable_parameters": sum(
                parameter.numel() for parameter in parameters if parameter.requires_grad
            ),
            "parameters_mib": sum(
                parameter.numel() * parameter.element_size() for parameter in parameters
            ) / 2**20,
            "buffers_mib": sum(
                buffer.numel() * buffer.element_size() for buffer in buffers
            ) / 2**20,
            "memory_bank_shape": list(bank.shape) if bank is not None else None,
            "memory_bank_mib": (
                bank.numel() * bank.element_size() / 2**20 if bank is not None else None
            ),
            "latency": {
                "predict_only": predict_summary,
                "decode_and_preprocess": _latency_summary(preprocessing_ms),
                "host_to_device_and_predict": accelerator_summary,
                "end_to_end": pipeline_summary,
            },
            "throughput_images_per_second": 1_000 / accelerator_summary["mean_ms"],
            "predict_only_images_per_second": 1_000 / predict_summary["mean_ms"],
            "end_to_end_images_per_second": 1_000 / pipeline_summary["mean_ms"],
            "cuda_memory": pipeline_memory,
            "predict_only_cuda_memory": predict_memory,
            "environment": {
                "python_version": sys.version.split()[0],
                "platform": platform.platform(),
                "torch_version": str(torch.__version__),
                "cuda_version": torch.version.cuda,
                "cudnn_version": torch.backends.cudnn.version(),
                "cudnn_benchmark": torch.backends.cudnn.benchmark,
                "cudnn_deterministic": torch.backends.cudnn.deterministic,
                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
                "float32_matmul_precision": torch.get_float32_matmul_precision(),
            },
            "energy_joules_per_image": None,
            "energy_note": "Not measured; latency alone is not an energy measurement",
        }
    finally:
        detector.train(was_training)

def _nvml_handle_for_cuda(nvml: Any, device: torch.device) -> tuple[Any, str]:
    """Match by CUDA UUID; never assume CUDA and NVML ordinals are identical."""
    properties = torch.cuda.get_device_properties(device)
    cuda_uuid = getattr(properties, "uuid", None)
    if cuda_uuid:
        uuid_text = cuda_uuid.decode("ascii") if isinstance(cuda_uuid, bytes) else str(cuda_uuid)
        if not uuid_text.startswith(("GPU-", "MIG-")):
            uuid_text = "GPU-" + uuid_text
        try:
            return nvml.nvmlDeviceGetHandleByUUID(uuid_text), "cuda_uuid"
        except Exception as error:
            raise RuntimeError(f"Cannot match the CUDA UUID to NVML: {error}") from error
    if nvml.nvmlDeviceGetCount() == 1:
        return nvml.nvmlDeviceGetHandleByIndex(0), "single_nvml_device"
    raise RuntimeError(
        "CUDA UUID is unavailable and multiple NVML GPUs exist; "
        "refusing to measure an ambiguous physical device"
    )


def _nvml_counter_energy_window(
    predict_once: Any, nvml: Any, handle: Any, duration_seconds: float,
) -> dict[str, Any]:
    """Counter units are mJ; inference failures propagate rather than being hidden."""
    method = "nvml_total_energy_counter"
    try:
        before = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
        if before < 0:
            raise ValueError("negative energy counter")
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    count = 0
    while count == 0 or time.perf_counter() - start < duration_seconds:
        predict_once()
        count += 1
    try:
        after = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    elapsed = time.perf_counter() - start
    if after <= before:
        return {
            "status": "unavailable", "method": method,
            "reason": "Energy counter did not advance or was reset during measurement",
        }
    joules = (after - before) / 1_000.0
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "counter_start_millijoules": before,
        "counter_end_millijoules": after,
        "power_samples": [],
    }


def _nvml_power_energy_window(
    predict_once: Any, nvml: Any, handle: Any,
    duration_seconds: float, sample_interval_seconds: float,
) -> dict[str, Any]:
    """Integrate measured GPU power, with explicit boundary samples and no TDP estimate."""
    method = "nvml_power_integration"

    def read_watts() -> float:
        watts = float(nvml.nvmlDeviceGetPowerUsage(handle)) / 1_000.0
        if not math.isfinite(watts) or watts <= 0:
            raise ValueError("NVML power reading must be positive and finite")
        return watts

    try:
        first_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    samples = [(start, first_watts)]
    errors = []
    stop = threading.Event()

    def sample_power() -> None:
        while not stop.wait(sample_interval_seconds):
            try:
                watts = read_watts()
                samples.append((time.perf_counter(), watts))
            except Exception as error:
                errors.append(str(error))
                return

    worker = threading.Thread(target=sample_power, daemon=True)
    worker.start()
    count = 0
    try:
        while count == 0 or time.perf_counter() - start < duration_seconds:
            predict_once()
            count += 1
            if errors:
                break
    finally:
        stop.set()
        worker.join()
    if errors:
        return {"status": "unavailable", "method": method, "reason": errors[0]}
    try:
        final_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    samples.append((time.perf_counter(), final_watts))
    elapsed = samples[-1][0] - start
    # Two endpoints alone are not adequate for the sampled-power fallback.
    if len(samples) < 3:
        return {
            "status": "unavailable", "method": method,
            "reason": "Too few power samples; increase the measurement window",
        }
    joules = sum(
        (right[0] - left[0]) * (left[1] + right[1]) * 0.5
        for left, right in zip(samples, samples[1:])
    )
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "power_sample_interval_seconds_requested": sample_interval_seconds,
        "power_sample_count": len(samples),
        "power_samples": [
            {"elapsed_seconds": timestamp - start, "power_watts": watts}
            for timestamp, watts in samples
        ],
    }


@torch.inference_mode()
def measure_anomaly_inference_energy(
    detector: Any, loader: Any, *, device: torch.device | str,
    repeats: int = 3, duration_seconds: float = 30.0,
    warmup_seconds: float = 5.0, sample_interval_seconds: float = 0.1,
    method: str = "auto",
) -> dict[str, Any]:
    """Measure device-wide GPU J/image in a separate sustained predict-only phase.

    The model and one preprocessed input are already GPU resident. Each complete
    predict is synchronized, and its output discarded. Includes GPU idle/boundary
    energy, excludes CPU, I/O and host-to-device energy. Raw total energy is not
    idle-subtracted and is not attributed to an individual process. Sensor errors
    yield an explicit unavailable/partial report; prediction errors propagate.
    """
    if not isinstance(repeats, int) or isinstance(repeats, bool) or repeats < 1:
        raise ValueError("Energy repeats must be a positive integer")
    for name, value in (
        ("duration_seconds", duration_seconds),
        ("sample_interval_seconds", sample_interval_seconds),
    ):
        if not math.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be positive and finite")
    if not math.isfinite(warmup_seconds) or warmup_seconds < 0:
        raise ValueError("warmup_seconds must be non-negative and finite")
    if method not in {"auto", "counter", "power"}:
        raise ValueError("Energy method must be auto, counter or power")
    if getattr(loader, "batch_size", None) != 1 or getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Energy requires batch_size=1 and num_workers=0")
    if not detector.is_fitted:
        raise RuntimeError("Restore the detector before measuring energy")
    device = torch.device(device)
    report = {
        "schema_version": 1,
        "status": "unavailable", "reason": None, "method": None,
        "metric": "device_wide_gpu_energy_per_inference",
        "scope": (
            "predict-only repeated synchronized inference; model and one preprocessed "
            "image are CUDA resident; excludes input transfer, CPU and loader work"
        ),
        "device": str(device), "batch_size": 1,
        "repeats_requested": repeats,
        "duration_seconds_per_repeat_requested": duration_seconds,
        "warmup_seconds_requested": warmup_seconds,
        "sample_interval_seconds_requested": sample_interval_seconds,
        "method_requested": method,
        "idle_subtracted": False, "process_attributed": False,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "total_inferences": 0, "runs": [], "warnings": [],
    }
    if device.type != "cuda" or not torch.cuda.is_available():
        report["reason"] = "CUDA GPU is required for NVML energy measurements"
        return report
    try:
        import pynvml
    except ImportError:
        report["reason"] = "Install nvidia-ml-py (import pynvml) to read GPU energy/power"
        return report
    try:
        pynvml.nvmlInit()
    except Exception as error:
        report["reason"] = f"NVML initialization failed: {error}"
        return report

    was_training = detector.training
    try:
        try:
            handle, mapping = _nvml_handle_for_cuda(pynvml, device)
            report["device_mapping"] = mapping
            uuid = pynvml.nvmlDeviceGetUUID(handle)
            report["nvml_gpu_uuid"] = uuid.decode() if isinstance(uuid, bytes) else str(uuid)
            name = pynvml.nvmlDeviceGetName(handle)
            report["gpu_name"] = name.decode() if isinstance(name, bytes) else str(name)
        except Exception as error:
            report["reason"] = f"NVML device mapping failed: {error}"
            return report
        try:
            processes = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
            report["gpu_compute_process_count_at_start"] = len(processes)
            if len(processes) > 1:
                report["warnings"].append(
                    "Multiple compute processes are visible on this GPU; "
                    "device-wide energy cannot be assigned exclusively to this model"
                )
        except Exception:
            report["gpu_compute_process_count_at_start"] = None
        try:
            report["power_limit_watts"] = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1_000.0
        except Exception:
            report["power_limit_watts"] = None
        detector.to(device).eval()
        try:
            batch = next(iter(loader))
        except StopIteration as error:
            raise ValueError("Energy measurement loader is empty") from error
        images = batch["image"].to(device)
        del batch
        if images.ndim != 4 or images.shape[0] != 1 or images.dtype != torch.float32:
            raise ValueError("Energy comparison requires one FP32 BCHW image")
        report["input_shape"] = list(images.shape)
        report["input_dtype"] = str(images.dtype)

        def predict_once() -> None:
            prediction = detector.predict(images)
            torch.cuda.synchronize(device)
            del prediction

        torch.cuda.synchronize(device)
        warmup_start = time.perf_counter()
        predict_once()
        while time.perf_counter() - warmup_start < warmup_seconds:
            predict_once()
        report["warmup_seconds_actual"] = time.perf_counter() - warmup_start
        chosen_method = "power" if method == "power" else "counter"
        for repeat in range(repeats):
            if chosen_method == "counter":
                run = _nvml_counter_energy_window(
                    predict_once, pynvml, handle, duration_seconds
                )
                # Auto chooses one successful backend on the first repeat only;
                # never average counter and integrated-power repeats together.
                if repeat == 0 and method == "auto" and run["status"] != "measured":
                    report["warnings"].append("Counter unavailable: " + str(run.get("reason")))
                    chosen_method = "power"
                    run = _nvml_power_energy_window(
                        predict_once, pynvml, handle,
                        duration_seconds, sample_interval_seconds,
                    )
            else:
                run = _nvml_power_energy_window(
                    predict_once, pynvml, handle,
                    duration_seconds, sample_interval_seconds,
                )
            run["repeat"] = repeat + 1
            report["runs"].append(run)
            if run["status"] != "measured":
                report["status"] = "partial" if repeat else "unavailable"
                report["reason"] = run.get("reason", "GPU energy sensor failed")
                return report
        runs = report["runs"]
        total_images = sum(run["inferences"] for run in runs)
        total_joules = sum(run["gpu_energy_joules"] for run in runs)
        total_seconds = sum(run["duration_seconds"] for run in runs)
        values = [run["energy_joules_per_image"] for run in runs]
        mean = sum(values) / len(values)
        report.update({
            "status": "measured", "reason": None,
            "method": runs[0]["method"],
            "total_inferences": total_images,
            "total_gpu_energy_joules": total_joules,
            "total_duration_seconds": total_seconds,
            "energy_joules_per_image": total_joules / total_images,
            "average_gpu_power_watts": total_joules / total_seconds,
            "energy_joules_per_image_std": (
                math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
                if len(values) > 1 else 0.0
            ),
        })
        return report
    finally:
        detector.train(was_training)
        pynvml.nvmlShutdown()


## Measurement protocol and validity checks

This is the same protocol used for PatchCore and TinyGLASS: **T4, FP32, batch 1, 10 warm-up and 50 timed iterations per scope**, CUDA synchronization, sequential loader, zero worker prefetch and no pinned-memory shortcut. Inference runs the complete teacher/student/autoencoder prediction path, including saved calibration and output normalization. There is no mixed-precision autocast, compilation or quantization.

RGB images remain in [0,1] after the fixed pose-A crop and bilinear antialiased resize to 256×256; ImageNet normalization is internal to the model. Applying it externally as well would change the evaluated model.

The verified September checkpoint predates `input_size` and `mixed_precision` in the current checkpoint configuration. The only allowed compatibility defaults are explicitly recorded: 256×256 is fixed by the manifest, and `mixed_precision=True` is the current **training-only constructor default**, not a change to inference or evidence of its original training precision. No other missing/unknown field or tensor is silently ignored.

- **Predict-only:** reuse one preprocessed device-resident image.
- **Host-to-device + predict:** time image transfer and the complete prediction.
- **End-to-end:** include the existing loader's RGB **and annotation** I/O and mask transforms. This is not a mask-free deployment pipeline.
- CUDA allocated/reserved baseline, total peak and incremental peak are reported separately; driver/process memory is not measured.

Use a fresh session containing only this detector. Compare the same named scope across models; latencies from separate benchmark phases cannot be subtracted to infer transfer cost. Unexpected validation counts or a manifest mismatch are recorded/warned, never used to filter samples. There is no full-dataset mask audit. **Energy is measured in a separate phase**, leaving the latency/memory protocol unchanged:
- 5 seconds of sustained warm-up, followed by 3 windows of at least 30 seconds.
- One preprocessed GPU-resident FP32 image; complete synchronized predictions; no loader or image-transfer work inside the window.
- NVML cumulative energy counter (mJ converted to J) is preferred. If unavailable or not advancing on the first window, auto switches to measured GPU power (mW converted to W), sampled every 0.1 s and integrated with the trapezoidal rule. Repeats never mix the two methods.
- J/image is total GPU energy divided by completed inferences, weighted across repeats; per-repeat values, standard deviation and power are also saved.
- These are whole-GPU readings including memory/idle/boundary energy, not process-attributed readings. CPU energy is not measured and idle is not subtracted.
- CUDA/NVML devices are matched by UUID rather than assuming ordinal 0 identifies the same GPU. Multiple visible compute processes trigger a warning.
- Unsupported sensors, counter resets and invalid power readings produce explicit unavailable/partial status and a null aggregate, never a nominal-TDP estimate.

Sensor APIs and Python bindings: [NVIDIA NVML](https://docs.nvidia.com/deploy/nvml-api/latest/api/group__nvmlDeviceQueries.html), [nvidia-ml-py](https://pypi.org/project/nvidia-ml-py/).


## Execute the benchmark

Restore the selected final model, build only the pose-A validation loader, measure inference latency/memory, measure GPU energy separately, and save the JSON/CSV reports. No training, test evaluation, calibration or score selection is executed.


In [ ]:
from __future__ import annotations

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("Select a Kaggle GPU session, preferably Tesla T4 to match the other benchmarks")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("highest")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(DEVICE)
    print(f"GPU: {gpu_name}")
    if "T4" not in gpu_name:
        print("WARNING: this is not a T4; do not compare these latencies directly with T4 results")

if CHECKPOINT_SOURCE == "google_drive":
    checkpoint_path, download_info = download_selected_checkpoint_from_drive()
else:
    if MODEL_CHECKPOINT_PATH is None:
        raise ValueError("Set MODEL_CHECKPOINT_PATH to your trusted final EfficientAD model.ckpt")
    checkpoint_path = Path(MODEL_CHECKPOINT_PATH).expanduser()
    download_info = {"source": "local", "path": str(checkpoint_path)}
detector, restore_info = load_frozen_efficientad(checkpoint_path, device=DEVICE)
print({
    "source_run_id": restore_info["source_run_id"],
    "input_size": detector.input_size,
    "channels": detector.channels,
    "calibration_mode": restore_info["calibration_mode"],
    "image_score_mode": restore_info["image_score_mode"],
    "legacy_defaulted_fields": restore_info["legacy_defaulted_fields"],
    "checkpoint_mib": restore_info["checkpoint_size_bytes"] / 2**20,
    "training_performed": False,
    "calibration_performed": False,
})


In [ ]:
from __future__ import annotations

DATASET_ROOT = (
    Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
    if DATASET_ROOT_OVERRIDE is not None else find_dataset_root()
)
preprocessor = WheelPreprocessor(PreprocessingConfig(
    resize=MODEL_INPUT_SIZE,
    resize_shorter_side=None,
    center_crop=None,
    pose_crop_enabled=True,
    pose_crops=POSE_CROPS,
    normalize_mean=None,
    normalize_std=None,
    augmentations_enabled=False,
))
benchmark_loader = build_dataloader(
    DATASET_ROOT, BENCHMARK_SPLIT,
    batch_size=BENCHMARK_BATCH_SIZE,
    num_workers=BENCHMARK_NUM_WORKERS,
    pin_memory=BENCHMARK_PIN_MEMORY,
    seed=RANDOM_SEED,
    preprocessing=preprocessor,
    camera_poses=CAMERA_POSES,
)
condition_counts = Counter(row["condition"] for row in benchmark_loader.dataset.rows)
if condition_counts != Counter({"clean": 296, "hole": 96}):
    print(f"WARNING: validation counts differ from the original pose-A dataset: {dict(condition_counts)}")
manifest_sha256 = sha256_file(DATASET_ROOT / "samples.csv")
source_manifest = restore_info["source_manifest_sha256"]
dataset_matches_source_manifest = None if source_manifest is None else manifest_sha256 == source_manifest
if dataset_matches_source_manifest is False:
    print("WARNING: attached samples.csv differs from the checkpoint dataset manifest; this will be recorded")
print({
    "dataset_root": str(DATASET_ROOT),
    "split": BENCHMARK_SPLIT,
    "camera_poses": CAMERA_POSES,
    "condition_counts": dict(condition_counts),
    "input_size": MODEL_INPUT_SIZE,
    "pose_crop": POSE_CROPS["A_overhead"],
    "manifest_sha256": manifest_sha256,
    "dataset_matches_source_manifest": dataset_matches_source_manifest,
})


In [ ]:
from __future__ import annotations

print("Benchmarking EfficientAD-S inference only; no fit or accuracy evaluation")
# Outputs of predict() are discarded inside the benchmark to avoid inflated peaks.
benchmark_report = benchmark_anomaly_inference(
    detector, benchmark_loader, device=DEVICE,
    warmup_runs=BENCHMARK_WARMUP_RUNS,
    timed_runs=BENCHMARK_TIMED_RUNS,
    split=BENCHMARK_SPLIT,
)
if benchmark_report["input_shape"] != [1, 3, *MODEL_INPUT_SIZE]:
    raise ValueError("Unexpected benchmark input shape")
if benchmark_report["input_dtype"] != "torch.float32":
    raise ValueError("This comparison requires FP32 inputs")
benchmark_report.update({
    **restore_info,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint_source": CHECKPOINT_SOURCE,
    "checkpoint_download": download_info,
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
    "preprocessing": {
        "resize": list(MODEL_INPUT_SIZE),
        "resize_shorter_side": None,
        "center_crop": None,
        "pose_crop_enabled": True,
        "normalize_mean": None,
        "normalize_std": None,
        "augmentations_enabled": False,
    },
    "dataset_root": str(DATASET_ROOT),
    "validation_condition_counts": dict(condition_counts),
    "manifest_sha256": manifest_sha256,
    "dataset_matches_source_manifest": dataset_matches_source_manifest,
    "pin_memory": BENCHMARK_PIN_MEMORY,
    "num_workers": BENCHMARK_NUM_WORKERS,
})
print(json.dumps(benchmark_report, indent=2, allow_nan=False))


In [ ]:
from __future__ import annotations

if ENERGY_ENABLED:
    print(
        f"Measuring GPU energy: {ENERGY_REPEATS} x {ENERGY_DURATION_SECONDS:g} s "
        f"after {ENERGY_WARMUP_SECONDS:g} s warm-up; predict-only, batch 1, FP32"
    )
    energy_report = measure_anomaly_inference_energy(
        detector, benchmark_loader, device=DEVICE,
        repeats=ENERGY_REPEATS,
        duration_seconds=ENERGY_DURATION_SECONDS,
        warmup_seconds=ENERGY_WARMUP_SECONDS,
        sample_interval_seconds=ENERGY_POWER_SAMPLE_INTERVAL_SECONDS,
        method=ENERGY_METHOD,
    )
else:
    energy_report = {
        "schema_version": 1, "status": "disabled",
        "reason": "ENERGY_ENABLED=False", "method": None,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "runs": [], "warnings": [],
    }
energy_report.update({
    "source_run_id": restore_info["source_run_id"],
    "checkpoint_path": restore_info["checkpoint_path"],
    "manifest_sha256": manifest_sha256,
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_performed": False,
    "calibration_performed": False,
})
benchmark_report["gpu_energy_measurement"] = energy_report
benchmark_report["energy_joules_per_image"] = energy_report["energy_joules_per_image"]
benchmark_report["energy_note"] = (
    "Measured device-wide GPU energy per predict-only inference via "
    + str(energy_report["method"])
    + "; excludes CPU/input transfer/loader; idle is not subtracted"
    if energy_report["status"] == "measured"
    else "GPU energy not measured completely: " + str(energy_report.get("reason"))
)
print({
    "energy_status": energy_report["status"],
    "energy_method": energy_report["method"],
    "gpu_energy_joules_per_image": energy_report["energy_joules_per_image"],
    "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
    "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
    "warnings": energy_report["warnings"],
    "reason": energy_report.get("reason"),
})
if energy_report["status"] not in {"measured", "disabled"}:
    print("WARNING: energy could not be measured reliably; inspect inference_energy_batch1.json")


In [ ]:
from __future__ import annotations

benchmark_run_id = (
    "efficientad_s_256x256_A_crop720_global_fixed70k_efficiency__"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
OUTPUT_DIR = OUTPUT_ROOT / benchmark_run_id
benchmark_report["benchmark_run_id"] = benchmark_run_id
report_path = save_json_atomic(benchmark_report, OUTPUT_DIR / "inference_benchmark_batch1.json")
energy_path = save_json_atomic(energy_report, OUTPUT_DIR / "inference_energy_batch1.json")
memory = benchmark_report["cuda_memory"] or {}
predict_memory = benchmark_report["predict_only_cuda_memory"] or {}
summary_row = {
    "benchmark_run_id": benchmark_run_id,
    "source_run_id": benchmark_report["source_run_id"],
    "model": benchmark_report["model"],
    "gpu": benchmark_report["gpu_name"],
    "input_height": MODEL_INPUT_SIZE[0],
    "input_width": MODEL_INPUT_SIZE[1],
    "batch_size": benchmark_report["batch_size"],
    "calibration_mode": benchmark_report["calibration_mode"],
    "image_score_mode": benchmark_report["image_score_mode"],
    "parameters": benchmark_report["total_parameters"],
    "parameters_mib": benchmark_report["parameters_mib"],
    "buffers_mib": benchmark_report["buffers_mib"],
    "checkpoint_mib": benchmark_report["checkpoint_size_bytes"] / 2**20,
    "predict_only_mean_ms": benchmark_report["latency"]["predict_only"]["mean_ms"],
    "predict_only_p95_ms": benchmark_report["latency"]["predict_only"]["p95_ms"],
    "host_to_device_and_predict_mean_ms": benchmark_report["latency"]["host_to_device_and_predict"]["mean_ms"],
    "end_to_end_mean_ms": benchmark_report["latency"]["end_to_end"]["mean_ms"],
    "predict_only_images_per_second": benchmark_report["predict_only_images_per_second"],
    "host_to_device_and_predict_images_per_second": benchmark_report["throughput_images_per_second"],
    "end_to_end_images_per_second": benchmark_report["end_to_end_images_per_second"],
    "cuda_peak_allocated_mib": memory.get("peak_allocated_mib"),
    "cuda_peak_reserved_mib": memory.get("peak_reserved_mib"),
    "cuda_incremental_peak_allocated_mib": memory.get("incremental_peak_allocated_mib"),
    "predict_only_cuda_peak_allocated_mib": predict_memory.get("peak_allocated_mib"),
    "energy_joules_per_image": benchmark_report["energy_joules_per_image"],
    "energy_status": energy_report["status"],
    "energy_method": energy_report["method"],
    "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
    "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
}
summary_path = OUTPUT_DIR / "inference_benchmark_batch1.csv"
with summary_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary_row))
    writer.writeheader()
    writer.writerow(summary_row)
print(f"JSON report: {report_path}")
print(f"CSV summary: {summary_path}")
print(f"Energy report: {energy_path}")
print("No training, accuracy evaluation, checkpoint overwrite or Drive upload was performed.")


## Saved outputs

- `inference_benchmark_batch1.json`: all latency distributions/throughputs, parameters, buffers and checkpoint size, CUDA baseline/total/incremental peaks, hardware/software metadata, restored calibration and preprocessing, checkpoint/dataset provenance and explicit legacy compatibility fields.
- `inference_benchmark_batch1.csv`: one-row comparison summary including GPU J/image, status/method, mean power and run-to-run standard deviation.
- `inference_energy_batch1.json`: energy scope, NVML device identification, all repeated windows, completed inference counts, energy/power and sensor samples or a precise unavailable reason.

Download the output folder from Kaggle. No accuracy results or checkpoints are resaved, and nothing is uploaded to Drive. GPU energy is reported only when all requested windows are measured successfully; unsupported sensors or incomplete measurements leave the aggregate null with an explicit reason.
